## Skorch vs. TensorFlow: Comparación para Clasificación de Datos Tabulares
### 1. ¿Qué es Skorch?
Skorch es un wrapper que permite utilizar redes neuronales de PyTorch con una API similar a la de scikit-learn, lo que facilita la integración con pipelines y herramientas como GridSearchCV.


Se usa cuando queremos aprovechar la flexibilidad de PyTorch pero con la simplicidad de scikit-learn.
### 2. ¿Qué es TensorFlow?
TensorFlow es una librería de aprendizaje profundo desarrollada por Google, con herramientas como Keras, que simplifican la creación y ajuste de redes neuronales.


s más adecuado para modelos de producción, entrenamiento distribuido y optimización con aceleración por hardware (GPU/TPU).


### 3. Ajuste de Hiperparámetros
Para ambos frameworks, podemos ajustar:

* Arquitectura de la red: Número de capas, neuronas por capa.

* Funciones de activación: ReLU, Sigmoid, etc.

* Tasa de aprendizaje (learning_rate).

* Tamaño del batch (batch_size).

* Número de épocas (epochs).

* Optimizador: Adam, SGD, etc.


Para encontrar la mejor combinación, usaremos GridSearchCV con Skorch y Keras Tuner para TensorFlow.

In [ ]:
!pip install torch
!pip install skorch
!pip install keras-tuner

In [20]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, GridSearchCV
from skorch import NeuralNetClassifier
from skorch.helper import SliceDataset
from skorch.callbacks import EarlyStopping

In [13]:
# Carga del dataset de simulaciones
data=pd.read_csv('./data//simulation_christmas_market_test1.csv',index_col=0)

# Creo la columna delay y elimino baseline duration y actual duration
data['delay']=(data['actual_duration']>data['baseline_duration'])*1
data.drop(columns=['baseline_duration','actual_duration'],inplace=True)
data.head(2)



,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration21,duration9,...,duration23,duration25,duration26,duration28,duration29,duration30,duration31,duration32,duration33,delay
0,0,59,339,12,22,253.103687,155.926174,142.722851,75.588264,201.357782,...,7.003812,8.561072,14.489850,9.963206,10.022028,10.495140,18.565477,7.529944,0,1
1,0,59,339,12,22,235.393636,156.772672,145.892140,75.400027,194.180942,...,8.801176,7.211925,15.010945,10.034144,11.097189,8.921021,18.532106,8.368341,0,1


In [17]:

# Split the dataset into feactures and class
X = data.drop(columns=['delay'])
y = data['delay']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---- Modelo con Skorch ----
class SkorchNet(nn.Module):
    def __init__(self, input_dim=33, hidden_dim=50, output_dim=2):
        super(SkorchNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        X = self.fc1(X)
        X = self.relu(X)
        X = self.fc2(X)
        return self.softmax(X)

net = NeuralNetClassifier(
    SkorchNet,
    module__input_dim=33,
    module__hidden_dim=50,
    module__output_dim=2,
    max_epochs=20,
    lr=0.01,
    optimizer=torch.optim.Adam,
    criterion=nn.CrossEntropyLoss,
    batch_size=32,
    iterator_train__shuffle=True,
    callbacks=[EarlyStopping(patience=3)]
)

# GridSearch para Skorch
params = {'lr': [0.01, 0.001], 'batch_size': [16, 32]}
gs = GridSearchCV(net, params, scoring='accuracy', cv=3)
gs.fit(X_train.values.astype(np.float32), y_train.values)

print("Best Skorch Model:", gs.best_params_)
print("Best Skorch Accuracy:", gs.best_score_)


  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3508       1.0000        0.3133  0.1703
      2        0.3133       1.0000        0.3133  0.0127
      3        0.3133       1.0000        0.3133  0.0136


/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczf

Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3508       1.0000        0.3133  0.0131
      2        0.3133       1.0000        0.3133  0.0123
      3        0.3133       1.0000        0.3133  0.0140
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3504       1.0000        0.3133  0.0119
      2        0.3133       1.0000        0.3133  0.0120
      3        0.3133       1.0000        0.3133  0.0121
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3133       1.0000        0.3133  0.0512


/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczf

      2        0.3133       1.0000        0.3133  0.0200
      3        0.3133       1.0000        0.3133  0.0115
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.6167       1.0000        0.3133  0.0120
      2        0.3133       1.0000        0.3133  0.0121
      3        0.3133       1.0000        0.3133  0.0116
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3133       1.0000        0.3133  0.0114
      2        0.3133       1.0000        0.3133  0.0118
      3        0.3133       1.0000        0.3133  0.0123
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczf

      3        0.3133       1.0000        0.3133  0.0070
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.3134       1.0000        0.3133  0.0082
      2        0.3133       1.0000        0.3133  0.0072
      3        0.3133       1.0000        0.3133  0.0069
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.8045       1.0000        0.3133  0.0073
      2        0.3133       1.0000        0.3133  0.0069
      3        0.3133       1.0000        0.3133  0.0196
Stopping since valid_loss has not improved in the last 3 epochs.
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.5432       1.0000        0.3133  0.0071
      2

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczf

      2        0.3133       1.0000        0.3133  0.0171
      3        0.3133       1.0000        0.3133  0.0190
Stopping since valid_loss has not improved in the last 3 epochs.
Best Skorch Model: {'batch_size': 16, 'lr': 0.01}
Best Skorch Accuracy: 1.0


/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_16761/2337275585.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)


In [21]:
# ---- Modelo con TensorFlow ----
def build_model(hp):
    model = keras.Sequential([
        layers.Dense(hp.Int('units', min_value=32, max_value=128, step=32), activation='relu'),
        layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', [0.01, 0.001])),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

from kerastuner.tuners import RandomSearch
tuner = RandomSearch(build_model, objective='val_accuracy', max_trials=5, executions_per_trial=1, directory='tuner_dir')

tuner.search(X_train, y_train, epochs=20, validation_split=0.2, batch_size=32)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Best TensorFlow Model: units={best_hps.get('units')}, learning_rate={best_hps.get('learning_rate')}")

# Evaluación final con los mejores hiperparámetros
best_model = tuner.hypermodel.build(best_hps)
best_model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)
test_loss, test_acc = best_model.evaluate(X_test, y_test)

print("Best TensorFlow Accuracy:", test_acc)

Trial 5 Complete [00h 00m 01s]
val_accuracy: 1.0

Best val_accuracy So Far: 1.0
Total elapsed time: 00h 00m 08s
Best TensorFlow Model: units=64, learning_rate=0.001
Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/20
20/20 ━━━━━━━━━━━━━━